<a href="https://colab.research.google.com/github/yusuf-codes10/deep-learining-project/blob/main/transfer_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data initialization

In [ ]:
# STEP 1 — Set Kaggle credentials (run this first)
import os
from getpass import getpass

os.environ['KAGGLE_USERNAME'] = input("Enter Kaggle username: ")
os.environ['KAGGLE_API_TOKEN'] = getpass("Enter Kaggle API key: ")

Enter Kaggle username: youcefcopy
Enter Kaggle API key: ··········


In [ ]:
# STEP 2 — Install Kaggle (run once per session)
!pip install -q kaggle

In [ ]:
# STEP 3 — Download dataset
!kaggle datasets download -d paultimothymooney/chest-xray-pneumonia

Dataset URL: https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia
License(s): other
100% 2.29G/2.29G [00:24<00:00, 99.0MB/s]



In [ ]:
# STEP 4 — Unzip dataset
!unzip -q chest-xray-pneumonia.zip

# Data Normaliation and Augmentaion

## 1- Laoding the images

In [ ]:
# ── Load all training images manually at 64x64 ──────────────────────────────
import numpy as np
import os
from tensorflow.keras.preprocessing.image import load_img, img_to_array, ImageDataGenerator

def load_images_from_folder(folder, label, target_size=(64, 64)):
    images, labels = [], []
    for fname in os.listdir(folder):
        fpath = os.path.join(folder, fname)
        try:
            img = load_img(fpath, target_size=target_size)
            images.append(img_to_array(img) / 255.0)
            labels.append(label)
        except:
            pass
    return images, labels

normal_imgs, normal_labels       = load_images_from_folder("chest_xray/train/NORMAL",    0)
pneumonia_imgs, pneumonia_labels = load_images_from_folder("chest_xray/train/PNEUMONIA", 1)

print(f"NORMAL:    {len(normal_imgs)}")
print(f"PNEUMONIA: {len(pneumonia_imgs)}")

NORMAL:    1341
PNEUMONIA: 3875


## 2- Data Augmentaion

In [ ]:
# ── Augment NORMAL only until it matches PNEUMONIA count ────────────────────
aug = ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    zoom_range=0.1
)

normal_imgs_aug    = list(normal_imgs)
normal_labels_aug  = list(normal_labels)
target_count       = len(pneumonia_imgs)
normal_array       = np.array(normal_imgs)

while len(normal_imgs_aug) < target_count:
    idx       = np.random.randint(0, len(normal_imgs))
    src       = normal_array[idx].reshape(1, 64, 64, 3)
    augmented = next(aug.flow(src, batch_size=1))[0]
    normal_imgs_aug.append(augmented)
    normal_labels_aug.append(0)

print(f"NORMAL after oversampling: {len(normal_imgs_aug)}")
print(f"PNEUMONIA:                 {len(pneumonia_imgs)}")

NORMAL after oversampling: 3875
PNEUMONIA:                 3875


## 3- Shuffling

In [ ]:
# ── Combine + shuffle ────────────────────────────────────────────────────────
X = np.array(normal_imgs_aug + pneumonia_imgs)
y = np.array(normal_labels_aug + pneumonia_labels)

idx  = np.random.permutation(len(X))
X, y = X[idx], y[idx]

print(f"Final dataset: {X.shape}")
print(f"NORMAL={sum(y==0)}, PNEUMONIA={sum(y==1)}")

Final dataset: (7750, 64, 64, 3)
NORMAL=3875, PNEUMONIA=3875


## 4- Splitting

In [ ]:
# ── Train/val split ──────────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape} | Val: {X_val.shape}")

Train: (6200, 64, 64, 3) | Val: (1550, 64, 64, 3)


# Transfer Model

## 1- 1- Base Model (Pre-trained on ImageNet)

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.applications import VGG16

# ── Load VGG16 without the top classification head ───────────────────────────
# include_top=False → removes the final Dense layers (we'll add our own)
# input_shape must match what we loaded: 64x64x3
# weights='imagenet' → uses features learned from 1.2M images
base_model = VGG16(
    weights='imagenet',
    include_top=False,
    input_shape=(64, 64, 3)
)

# ── Freeze all base layers first ─────────────────────────────────────────────
# We don't want to destroy the pre-trained ImageNet features during early training
base_model.trainable = False

print(f"Base model layers: {len(base_model.layers)}")
base_model.summary()

58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Base model layers: 19


Model: "vgg16"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 64, 64, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv1 (Conv2D)           │ (None, 64, 64, 64)     │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv2 (Conv2D)           │ (None, 64, 64, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_pool (MaxPooling2D)      │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv1 (Conv2D)           │ (None, 32, 32, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv2 (Conv2D)           │ (None, 32, 32, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_pool (MaxPooling2D)      │ (None, 16, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv1 (Conv2D)           │ (None, 16, 16, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv2 (Conv2D)           │ (None, 16, 16, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv3 (Conv2D)           │ (None, 16, 16, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_pool (MaxPooling2D)      │ (None, 8, 8, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv1 (Conv2D)           │ (None, 8, 8, 512)      │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv2 (Conv2D)           │ (None, 8, 8, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv3 (Conv2D)           │ (None, 8, 8, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_pool (MaxPooling2D)      │ (None, 4, 4, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv1 (Conv2D)           │ (None, 4, 4, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv2 (Conv2D)           │ (None, 4, 4, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv3 (Conv2D)           │ (None, 4, 4, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_pool (MaxPooling2D)      │ (None, 2, 2, 512)      │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,714,688 (56.13 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 14,714,688 (56.13 MB)

## 2-  Model Architecture (Custom Head on top of VGG16)

In [ ]:
from tensorflow.keras import regularizers

l2 = regularizers.l2(1e-5)

# ── Build the full model ──────────────────────────────────────────────────────
# Same structure philosophy as cnn_v3: Flatten → Dense(128) + L2 → Dropout → Sigmoid
inputs = keras.Input(shape=(64, 64, 3))

x = base_model(inputs, training=False)   # training=False → BN layers stay frozen

x = keras.layers.GlobalAveragePooling2D()(x)   # more efficient than Flatten for transfer

# ✅ L2 only on Dense (same as cnn_v3)
x = keras.layers.Dense(128, activation='relu', kernel_regularizer=l2)(x)

x = keras.layers.Dropout(0.6)(x)   # 🔥 same dropout rate as cnn_v3

outputs = keras.layers.Dense(1, activation='sigmoid')(x)

transfer_model = keras.Model(inputs, outputs, name='vgg16_transfer')

transfer_model.summary()

Model: "vgg16_transfer"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 64, 64, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ vgg16 (Functional)              │ (None, 2, 2, 512)      │    14,714,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 512)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        65,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,780,481 (56.38 MB)

 Trainable params: 65,793 (257.00 KB)

 Non-trainable params: 14,714,688 (56.13 MB)

## 3- Compiling

In [ ]:
# ── Compile with a low learning rate ─────────────────────────────────────────
# Lower lr than default adam (1e-3) → avoids overwriting frozen features too fast
transfer_model.compile(
    loss='binary_crossentropy',
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    metrics=['accuracy']
)

## 4-Fitting (Frozen base — Phase 1)

In [ ]:
# ── Phase 1: Train only the custom head ──────────────────────────────────────
# Base model is frozen → only Dense layers are updated
# This is fast and stabilizes the new head before fine-tuning
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

transfer_history_phase1 = transfer_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=15,
    batch_size=32,
    callbacks=[early_stop]
    # ❗ no class_weight — data is already balanced (same as cnn_v3)
)

Epoch 1/15
194/194 ━━━━━━━━━━━━━━━━━━━━ 16s 52ms/step - accuracy: 0.6360 - loss: 0.6350 - val_accuracy: 0.8400 - val_loss: 0.5052
Epoch 2/15
194/194 ━━━━━━━━━━━━━━━━━━━━ 4s 22ms/step - accuracy: 0.8050 - loss: 0.4759 - val_accuracy: 0.8619 - val_loss: 0.4122
Epoch 3/15
194/194 ━━━━━━━━━━━━━━━━━━━━ 4s 22ms/step - accuracy: 0.8566 - loss: 0.4001 - val_accuracy: 0.8871 - val_loss: 0.3568
Epoch 4/15
194/194 ━━━━━━━━━━━━━━━━━━━━ 5s 23ms/step - accuracy: 0.8739 - loss: 0.3539 - val_accuracy: 0.8987 - val_loss: 0.3171
Epoch 5/15
194/194 ━━━━━━━━━━━━━━━━━━━━ 5s 24ms/step - accuracy: 0.8889 - loss: 0.3149 - val_accuracy: 0.9065 - val_loss: 0.2885
Epoch 6/15
194/194 ━━━━━━━━━━━━━━━━━━━━ 5s 28ms/step - accuracy: 0.9045 - loss: 0.2873 - val_accuracy: 0.9129 - val_loss: 0.2642
Epoch 7/15
194/194 ━━━━━━━━━━━━━━━━━━━━ 5s 23ms/step - accuracy: 0.9105 - loss: 0.2617 - val_accuracy: 0.9206 - val_loss: 0.2445
Epoch 8/15
194/194 ━━━━━━━━━━━━━━━━━━━━ 5s 23ms/step - accuracy: 0.9144 - loss: 0.2453 - val_acc

## 5- Fine-Tuning (Unfreeze top layers — Phase 2)

In [ ]:
# ── Phase 2: Unfreeze the last few conv blocks of VGG16 ──────────────────────
# We only unfreeze block5 (last conv block) — earlier layers hold generic features
# Fine-tuning with a very low lr to slightly adapt VGG features to chest X-rays
base_model.trainable = True

# Freeze everything except block5_conv1 onwards
for layer in base_model.layers:
    if layer.name.startswith("block5"):
        layer.trainable = True
    else:
        layer.trainable = False

print("Trainable layers:")
for layer in transfer_model.layers:
    print(f"  {layer.name}: trainable={layer.trainable}")

Trainable layers:
  input_layer_1: trainable=True
  vgg16: trainable=True
  global_average_pooling2d: trainable=True
  dense: trainable=True
  dropout: trainable=True
  dense_1: trainable=True


In [ ]:
# ── Re-compile after unfreezing (mandatory!) ──────────────────────────────────
# Very low lr → avoids destroying pre-trained weights during fine-tuning
transfer_model.compile(
    loss='binary_crossentropy',
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),   # 🔥 10x lower than Phase 1
    metrics=['accuracy']
)

transfer_history_phase2 = transfer_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=15,
    batch_size=32,
    callbacks=[early_stop]
)

Epoch 1/15
194/194 ━━━━━━━━━━━━━━━━━━━━ 16s 59ms/step - accuracy: 0.9584 - loss: 0.1175 - val_accuracy: 0.9761 - val_loss: 0.0739
Epoch 2/15
194/194 ━━━━━━━━━━━━━━━━━━━━ 8s 41ms/step - accuracy: 0.9744 - loss: 0.0723 - val_accuracy: 0.9748 - val_loss: 0.0692
Epoch 3/15
194/194 ━━━━━━━━━━━━━━━━━━━━ 7s 38ms/step - accuracy: 0.9847 - loss: 0.0490 - val_accuracy: 0.9813 - val_loss: 0.0555
Epoch 4/15
194/194 ━━━━━━━━━━━━━━━━━━━━ 8s 39ms/step - accuracy: 0.9858 - loss: 0.0419 - val_accuracy: 0.9594 - val_loss: 0.1097
Epoch 5/15
194/194 ━━━━━━━━━━━━━━━━━━━━ 8s 41ms/step - accuracy: 0.9865 - loss: 0.0368 - val_accuracy: 0.9755 - val_loss: 0.0688
Epoch 6/15
194/194 ━━━━━━━━━━━━━━━━━━━━ 8s 43ms/step - accuracy: 0.9916 - loss: 0.0264 - val_accuracy: 0.9813 - val_loss: 0.0539
Epoch 7/15
194/194 ━━━━━━━━━━━━━━━━━━━━ 8s 43ms/step - accuracy: 0.9935 - loss: 0.0224 - val_accuracy: 0.9832 - val_loss: 0.0519
Epoch 8/15
194/194 ━━━━━━━━━━━━━━━━━━━━ 8s 41ms/step - accuracy: 0.9942 - loss: 0.0168 - val_acc

## 6- Test and Evaluation

In [ ]:
# ── Graph ─────────────────────────────────────────────────────────────────────
import pandas as pd
import matplotlib.pyplot as plt

# Combine both phases into one history for a full picture
combined_history = {}
for key in transfer_history_phase1.history:
    combined_history[key] = (
        transfer_history_phase1.history[key] +
        transfer_history_phase2.history[key]
    )

pd.DataFrame(combined_history).plot(figsize=(15, 8))
plt.grid(True)
plt.gca().set_ylim(0, 1)
plt.axvline(x=len(transfer_history_phase1.history['loss']) - 1,
            color='gray', linestyle='--', label='Fine-tuning starts')
plt.title('VGG16 Transfer | Phase 1 + Phase 2 Fine-Tuning')
plt.legend()
plt.show()

In [ ]:
# ── Evaluation with threshold 0.7 ─────────────────────────────────────────────
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

# Reuse test set loaded the same way as cnn_v3
transfer_preds    = transfer_model.predict(X_test)
predicted_transfer = (transfer_preds > 0.7).astype(int).flatten()   # 🔥 threshold 0.7

print(classification_report(y_test, predicted_transfer, target_names=['NORMAL', 'PNEUMONIA']))

cm = confusion_matrix(y_test, predicted_transfer)
ConfusionMatrixDisplay(cm, display_labels=['NORMAL', 'PNEUMONIA']).plot(cmap='Blues')
plt.title('VGG16 Transfer | Threshold 0.7')
plt.show()

## 7- Saving the Model

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

save_path = "/content/drive/MyDrive/messi"
os.makedirs(save_path, exist_ok=True)

transfer_model.save(f"{save_path}/vgg16_transfer_final.h5")
print("Saved!")